In [7]:
import yaml

def get_config():
    with open('config.yaml', encoding='utf-8') as f:
        config = yaml.safe_load(f)
        return config

config = get_config()

In [5]:
%pip install h5py

  Using cached h5py-3.15.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached h5py-3.15.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (4.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import h5py
import numpy as np
from pathlib import Path

In [10]:
%pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 11.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 10.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 11.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import matplotlib.pyplot as plt

In [31]:
root = '/home/hyunjin/RBY1_migration/rby1_ws/rby1-data-collection/Demo/'
task_name = config['conversion_task_name']
demo_root = os.path.join(root, task_name)

RAW_H5_DIR = Path(demo_root)
# print(RAW_H5_DIR / "demo_103.h5")
# h5_files = sorted(list(RAW_H5_DIR.glob("*.h5")))
# print(h5_files)
# h5_file_path = h5_files[-1]
h5_file_path = RAW_H5_DIR / "demo_104.h5"

In [32]:
# Create a tree visualization
def create_tree_visualization(file_path):
    """Create a tree-like visualization of the HDF5 structure"""
    
    def tree_structure(name, obj, prefix="", is_last=True):
        """Recursively create tree structure"""
        connector = "└── " if is_last else "├── "
        
        if isinstance(obj, h5py.Group):
            print(f"{prefix}{connector}📁 {name.split('/')[-1] if '/' in name else name}")
        elif isinstance(obj, h5py.Dataset):
            shape_str = f"{obj.shape}" if len(obj.shape) > 0 else "scalar"
            print(f"{prefix}{connector}📄 {name.split('/')[-1] if '/' in name else name} {shape_str} [{obj.dtype}]")
    
    with h5py.File(file_path, 'r') as f:
        print("\n" + "="*70)
        print("Tree Structure Visualization:")
        print("="*70)
        print(f"\n📦 {file_path.name}")
        
        items = list(f.items())
        for i, (name, obj) in enumerate(items):
            is_last = (i == len(items) - 1)
            tree_structure(name, obj, "", is_last)
            
            # If it's a group, show its children
            if isinstance(obj, h5py.Group):
                sub_items = list(obj.items())
                for j, (sub_name, sub_obj) in enumerate(sub_items):
                    sub_is_last = (j == len(sub_items) - 1)
                    sub_prefix = "    " if is_last else "│   "
                    tree_structure(sub_name, sub_obj, sub_prefix, sub_is_last)

create_tree_visualization(h5_file_path)


Tree Structure Visualization:

📦 demo_104.h5
├── 📁 head_depth
├── 📁 head_rgb
├── 📁 pointclouds
└── 📁 samples


In [33]:
with h5py.File(h5_file_path, 'r') as f:
    # if 'head_rgb' in f and 'image' in f['head_rgb']:
    if 'head_rgb' in f:
        if 'image' in f['head_rgb']:
            print("Image available")
        else:
            print("Image unavailable")

Image unavailable


In [34]:
# Display RGB and Depth images
# h5_file_path = Path('/media/nvidia/T7/Demo/demo_5.h5')

with h5py.File(h5_file_path, 'r') as f:
    if 'head_rgb' in f and 'image' in f['head_rgb']:
        rgb_images = f['head_rgb/image'][:]
        rgb_times = f['head_rgb/time'][:]
        print(f"✅ RGB Images: {rgb_images.shape}")
        print(f"   Frame count: {len(rgb_images)}")
        print(f"   Time range: {rgb_times[0]:.2f}s to {rgb_times[-1]:.2f}s" if len(rgb_times) > 0 else "   No timestamps")
    else:
        rgb_images = None
        print("❌ No RGB images found")
    
    if 'head_depth' in f and 'image' in f['head_depth']:
        depth_images = f['head_depth/image'][:]
            
        depth_times = f['head_depth/time'][:]
        print(f"\n✅ Depth Images: {depth_images.shape}")
        print(f"   Frame count: {len(depth_images)}")
        print(f"   Time range: {depth_times[0]:.2f}s to {depth_times[-1]:.2f}s" if len(depth_times) > 0 else "   No timestamps")
        print(f"   Depth range: {depth_images.min()} to {depth_images.max()} mm")
    else:
        depth_images = None
        print("\n❌ No depth images found")
    
    # Display a sample frame
    if rgb_images is not None and len(rgb_images) > 0:
        sample_idx = len(rgb_images) // 2  # middle frame
        
        fig, axes = plt.subplots(1, 2 if depth_images is not None else 1, figsize=(12, 6))
        
        if depth_images is not None:
            # RGB
            axes[0].imshow(rgb_images[sample_idx])
            axes[0].set_title(f'RGB Image (Frame {sample_idx})')
            axes[0].axis('off')
            
            # Depth (colorized)
            depth_colorized = depth_images[sample_idx].astype(float)
            # Normalize for visualization
            depth_colorized[depth_colorized == 0] = np.nan  # Remove zeros
            im = axes[1].imshow(depth_colorized, cmap='jet')
            axes[1].set_title(f'Depth Image (Frame {sample_idx})')
            axes[1].axis('off')
            plt.colorbar(im, ax=axes[1], label='Depth (mm)')
        else:
            axes.imshow(rgb_images[sample_idx])
            axes.set_title(f'RGB Image (Frame {sample_idx})')
            axes.axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("\nNo images to display")

    # Display all frame
    if rgb_images is not None and len(rgb_images) > 0:
        for sample_idx in range(110,136,3):
            fig, axes = plt.subplots(1, 2 if depth_images is not None else 1, figsize=(12, 6))
            
            if depth_images is not None:
                # RGB
                axes[0].imshow(rgb_images[sample_idx])
                axes[0].set_title(f'RGB Image (Frame {sample_idx})')
                axes[0].axis('off')
                
                # Depth (colorized)
                depth_colorized = depth_images[sample_idx].astype(float)
                # Normalize for visualization
                depth_colorized[depth_colorized == 0] = np.nan  # Remove zeros
                im = axes[1].imshow(depth_colorized, cmap='jet')
                axes[1].set_title(f'Depth Image (Frame {sample_idx})')
                axes[1].axis('off')
                plt.colorbar(im, ax=axes[1], label='Depth (mm)')
            else:
                axes.imshow(rgb_images[sample_idx])
                axes.set_title(f'RGB Image (Frame {sample_idx})')
                axes.axis('off')
            
            plt.tight_layout()
            plt.show()
        else:
            print("\nNo images to display")

❌ No RGB images found

❌ No depth images found

No images to display


In [35]:
# Load and compare robot position vs target joints
with h5py.File(h5_file_path, 'r') as f:
    if 'samples' in f and len(f['samples'].keys()) > 0:
        print("="*70)
        print("Robot Joint Data Analysis")
        print("="*70)
        
        if 'robot_position' in f['samples']:
            robot_pos = f['samples/robot_position'][:]
            print(f"\n✅ Robot Position (Current): {robot_pos.shape}")
            print(f"   First sample: {robot_pos[0] if len(robot_pos) > 0 else 'No data'}")
        
        if 'robot_target_cartesian' in f['samples']:
            target_cart = f['samples/robot_target_cartesian'][:]
            print(f"\n✅ Robot Target (Cartesian): {target_cart.shape}")
            print(f"   Format: [right_xyz(3), left_xyz(3), torso_xyz(3)]")
            print(f"   First sample: {target_cart[0] if len(target_cart) > 0 else 'No data'}")
            for i in range(len(target_cart)):
                print(f"   Sample {i}: {target_cart[i]}")  
        
        if 'robot_target_joints' in f['samples']:
            target_joints = f['samples/robot_target_joints'][:]
            print(f"\n✅ Robot Target (Joint Angles): {target_joints.shape}")
            print(f"   First sample: {target_joints[0] if len(target_joints) > 0 else 'No data'}")
            for i in range(len(target_joints)):
                print(f"   Sample {i}: {target_joints[i]}")
            
            # Check if IK succeeded (not all NaN)
            if len(target_joints) > 0:
                valid_samples = ~np.isnan(target_joints).all(axis=1)
                print(f"   Valid IK solutions: {valid_samples.sum()} / {len(target_joints)}")
                
                if valid_samples.sum() > 0 and 'robot_position' in f['samples']:
                    # Compute difference between target and actual
                    diff = target_joints[valid_samples] - robot_pos[valid_samples]
                    print(f"\n📊 Joint Angle Errors (target - actual):")
                    print(f"   Mean error: {np.rad2deg(np.nanmean(np.abs(diff), axis=0))} degrees")
                    print(f"   Max error:  {np.rad2deg(np.nanmax(np.abs(diff), axis=0))} degrees")
    else:
        print("No sample data found in the file.")

No sample data found in the file.


현재 문제점: 이미지와 액션 데이터 모두 안보임
데이터가 문제인가 코드가 문제인가